In [2]:
"""
recovery_manager.py
Simulates an undo/redo recovery manager for a database after a crash.

Log record formats (one per line, read from log_input.txt):
    <T1 start>
    <T1, X, 100, 150>     -> <transaction, variable, old value (v1), new value (v2)>
    <T1 commit>
    <T1 abort>

Recovery rules:
  * committed  (start + commit)      -> REDO : write new value (v2), scanning forward
  * incomplete (start, no commit/abort) -> UNDO : write old value (v1), scanning backward
  * aborted    (start + abort)       -> also undone (idempotent) so that any
                                       partially-rolled-back writes on disk are cleaned up
Undo is performed first (backward scan), then redo (forward scan).
"""

import re
import sys

LOG_FILE = "log_input.txt"

# State of the database on disk at the moment of the crash. Some changes of
# uncommitted transactions may have been flushed (dirty), and some changes of
# committed transactions may not have reached the disk yet.
DISK_STATE_AT_CRASH = {"X": 175, "Y": 200, "Z": 350}

RE_STATUS = re.compile(r"^<\s*(\w+)\s+(start|commit|abort)\s*>$", re.IGNORECASE)
RE_UPDATE = re.compile(r"^<\s*(\w+)\s*,\s*(\w+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*>$")


def parse_log(path):
    """Return a list of log records as tuples."""
    records = []
    with open(path) as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            m = RE_STATUS.match(line)
            if m:
                records.append((m.group(2).lower(), m.group(1)))
                continue
            m = RE_UPDATE.match(line)
            if m:
                t, var, old, new = m.groups()
                records.append(("update", t, var, int(old), int(new)))
                continue
            print(f"Warning: skipping malformed log line {lineno}: {line}", file=sys.stderr)
    return records


def classify(records):
    """Split transactions into committed, aborted and incomplete sets."""
    started, committed, aborted = [], set(), set()
    for rec in records:
        kind = rec[0]
        if kind == "start":
            started.append(rec[1])
        elif kind == "commit":
            committed.add(rec[1])
        elif kind == "abort":
            aborted.add(rec[1])
    incomplete = [t for t in started if t not in committed and t not in aborted]
    return committed, aborted, incomplete


def recover(records, db):
    committed, aborted, incomplete = classify(records)
    undo_set = set(incomplete) | aborted

    print("Committed  transactions (REDO):", sorted(committed))
    print("Aborted    transactions (cleanup UNDO):", sorted(aborted))
    print("Incomplete transactions (UNDO):", sorted(incomplete))
    print()

    # Phase 1: UNDO - scan the log backward, restore old values (v1)
    print("--- UNDO phase (backward scan) ---")
    for rec in reversed(records):
        if rec[0] == "update" and rec[1] in undo_set:
            _, t, var, old, new = rec
            db[var] = old
            print(f"UNDO {t}: {var} = {old}")

    # Phase 2: REDO - scan the log forward, apply new values (v2)
    print("\n--- REDO phase (forward scan) ---")
    for rec in records:
        if rec[0] == "update" and rec[1] in committed:
            _, t, var, old, new = rec
            db[var] = new
            print(f"REDO {t}: {var} = {new}")
    return db


def main():
    # Ignore command line arguments like '-f' when running in an interactive environment (notebook)
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        path = sys.argv[1]
    else:
        path = LOG_FILE

    records = parse_log(path)
    db = dict(DISK_STATE_AT_CRASH)

    print("=== Recovery Manager Simulation ===")
    print("Disk state at crash:", db, "\n")
    db = recover(records, db)
    print("\n=== Final values after recovery ===")
    for var in sorted(db):
        print(f"{var} = {db[var]}")


if __name__ == "__main__":
    main()

=== Recovery Manager Simulation ===
Disk state at crash: {'X': 175, 'Y': 200, 'Z': 350} 

Committed  transactions (REDO): ['T1', 'T2']
Aborted    transactions (cleanup UNDO): ['T4']
Incomplete transactions (UNDO): ['T3']

--- UNDO phase (backward scan) ---
UNDO T4: X = 150
UNDO T3: Z = 300

--- REDO phase (forward scan) ---
REDO T1: X = 150
REDO T2: Y = 250
REDO T2: Y = 275

=== Final values after recovery ===
X = 150
Y = 275
Z = 300
